In [1]:
"""
ViT-Small/16 fine-tuning — Layer 2 candidate, matched to the MobileNetV3 baseline.

Same splits (loaded from splits.csv), same colour constancy, same weight cap,
same label smoothing, same two-stage schedule, same calibration + eval. The
ONLY variable is the model, so the comparison to MobileNetV3 (and to the
DINOv2 frozen+MLP run, which underperformed) is honest.

Colab, GPU runtime. Run MobileNetV3 Cells 1-3 first in the SAME session (or
a prior session that already wrote splits.csv + the image cache) so this
reuses the identical data — that's what makes the comparison valid.

Full fine-tune, not frozen+linear-probe: a self-supervised DINOv2 backbone
with a frozen encoder underperformed MobileNetV3 here, which is a strong
sign that lesion classification needs backbone adaptation, not just a probe
on generic features. So this uses a supervised ImageNet-21k ViT and
fine-tunes the whole thing, same as MobileNetV3's stage 2.
"""

"\nViT-Small/16 fine-tuning — Layer 2 candidate, matched to the MobileNetV3 baseline.\n\nSame splits (loaded from splits.csv), same colour constancy, same weight cap,\nsame label smoothing, same two-stage schedule, same calibration + eval. The\nONLY variable is the model, so the comparison to MobileNetV3 (and to the\nDINOv2 frozen+MLP run, which underperformed) is honest.\n\nColab, GPU runtime. Run MobileNetV3 Cells 1-3 first in the SAME session (or\na prior session that already wrote splits.csv + the image cache) so this\nreuses the identical data — that's what makes the comparison valid.\n\nFull fine-tune, not frozen+linear-probe: a self-supervised DINOv2 backbone\nwith a frozen encoder underperformed MobileNetV3 here, which is a strong\nsign that lesion classification needs backbone adaptation, not just a probe\non generic features. So this uses a supervised ImageNet-21k ViT and\nfine-tunes the whole thing, same as MobileNetV3's stage 2.\n"

## CELL 1: setup

In [3]:
# ============================================================ CELL 1: setup
import os, glob, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import kagglehub
DATA = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("dataset:", DATA)

!pip install -q timm
import timm

SEED = 42
OUT = "/content/drive/MyDrive/amsdds"
CACHE = "/content/ham_cache"
os.makedirs(OUT, exist_ok=True)
os.makedirs(CACHE, exist_ok=True)

IMG_SIZE = 224
BATCH = 32
WEIGHT_CAP = 2.0
USE_COLOR_CONSTANCY = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE)

Mounted at /content/drive


100%|██████████| 5.20G/5.20G [00:41<00:00, 133MB/s]

Extracting files...


dataset: /root/.cache/kagglehub/datasets/kmader/skin-cancer-mnist-ham10000/versions/2
device: cuda


## CELL 2: reuse the SAME splits as MobileNetV3

In [4]:
# ============================================ CELL 2: reuse the SAME splits
src = {os.path.splitext(os.path.basename(p))[0]: p
       for p in glob.glob(f"{DATA}/**/*.jpg", recursive=True)}

splits_csv = f"{OUT}/splits.csv"
assert os.path.exists(splits_csv), (
    "splits.csv not found -- run the MobileNetV3 script's Cell 2 first "
    "(or copy splits.csv into OUT) so Layer 1 and Layer 2 share identical "
    "train/val/test splits. Comparing on different splits is not valid."
)
df = pd.read_csv(splits_csv)
print("loaded existing splits — comparison is apples-to-apples")

CLASSES = sorted(df.dx.unique())
df["y"] = df.dx.map({c: i for i, c in enumerate(CLASSES)})
df["src"] = df.image_id.map(src)
df["path"] = df.image_id.map(lambda i: f"{CACHE}/{i}.jpg")
assert df.src.isna().sum() == 0, "some images not found"
print(df.split.value_counts().to_dict())

loaded existing splits — comparison is apples-to-apples
{'train': 7116, 'val': 1458, 'test': 1441}


## CELL 3: reuse the preprocessed image cache

In [5]:
# ================================ CELL 3: one-time preprocessed image cache
# Same colour-constancy cache as MobileNetV3 -- if that script already ran
# in this OUT dir, this is a no-op (files already exist on local disk).
def shades_of_gray(arr, power=6):
    a = arr.astype(np.float32)
    vec = np.power(np.mean(np.power(a, power), axis=(0, 1)), 1.0 / power)
    vec = vec / (np.sqrt(np.sum(vec ** 2)) + 1e-8)
    return np.clip(a / (vec * np.sqrt(3) + 1e-8), 0, 255).astype(np.uint8)


def build_one(row):
    dst = row.path
    if os.path.exists(dst):
        return
    img = Image.open(row.src).convert("RGB")
    w, h = img.size
    s = 256 / min(w, h)
    img = img.resize((round(w * s), round(h * s)), Image.BICUBIC)
    if USE_COLOR_CONSTANCY:
        img = Image.fromarray(shades_of_gray(np.array(img)))
    img.save(dst, quality=95)


need = sum(not os.path.exists(p) for p in df.path)
if need:
    from concurrent.futures import ThreadPoolExecutor
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex:
        list(ex.map(build_one, [r for r in df.itertuples()]))
    print(f"cached {need} images in {time.time()-t0:.0f}s")
else:
    print("image cache already built")

cached 10015 images in 143s


## CELL 4: datasets + transforms (identical to MobileNetV3)

In [6]:
# ============================================ CELL 4: datasets + transforms
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0), ratio=(0.85, 1.18)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomApply([T.RandomRotation(30)], p=0.5),
    T.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.03),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.25),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize(256), T.CenterCrop(IMG_SIZE),
    T.ToTensor(), T.Normalize(MEAN, STD),
])


class HAM(Dataset):
    def __init__(self, frame, tf):
        self.f = frame.reset_index(drop=True); self.tf = tf

    def __len__(self):
        return len(self.f)

    def __getitem__(self, i):
        r = self.f.iloc[i]
        return self.tf(Image.open(r.path).convert("RGB")), int(r.y)


tr, va, te = (df[df.split == s] for s in ("train", "val", "test"))
dl_tr = DataLoader(HAM(tr, train_tf), BATCH, shuffle=True, num_workers=4,
                   pin_memory=True, drop_last=True, persistent_workers=True)
dl_va = DataLoader(HAM(va, eval_tf), 64, num_workers=4, pin_memory=True)
dl_te = DataLoader(HAM(te, eval_tf), 64, num_workers=4, pin_memory=True)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## CELL 5: model + loss
`vit_small_patch16_224.augreg_in21k_ft_in1k` -- supervised ImageNet-21k
pretrain, fine-tuned on ImageNet-1k. Full fine-tune (both stages), same
weight-capped loss + label smoothing as MobileNetV3.

In [7]:
# ==================================================== CELL 5: model + loss
MODEL_NAME = "vit_small_patch16_224.augreg_in21k_ft_in1k"
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES))
model = model.to(DEVICE)

counts = np.bincount(tr.y.values, minlength=len(CLASSES)).astype(np.float32)
w = 1.0 / np.sqrt(counts)
w = np.clip(w / w.min(), 1.0, WEIGHT_CAP)
print("class weights:", dict(zip(CLASSES, np.round(w, 2))))
lossfn = nn.CrossEntropyLoss(weight=torch.tensor(w).to(DEVICE), label_smoothing=0.05)

scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
from sklearn.metrics import f1_score


@torch.no_grad()
def evaluate(loader, return_logits=False):
    model.eval()
    L, Y = [], []
    for x, y in loader:
        with torch.autocast("cuda", torch.float16, enabled=DEVICE == "cuda"):
            L.append(model(x.to(DEVICE)).float().cpu())
        Y.append(y)
    L, Y = torch.cat(L), torch.cat(Y)
    f1 = f1_score(Y.numpy(), L.argmax(1).numpy(), average="macro")
    return (f1, L, Y) if return_logits else f1


def run_epochs(n, opt, sched, tag):
    global best_f1, best_state
    for ep in range(n):
        model.train(); t0 = time.time()
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", torch.float16, enabled=DEVICE == "cuda"):
                loss = lossfn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        f1 = evaluate(dl_va)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"{tag} ep {ep:2d}  val macro-F1 {f1:.4f}  ({time.time()-t0:.0f}s)")

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

class weights: {'akiec': np.float32(2.0), 'bcc': np.float32(2.0), 'bkl': np.float32(2.0), 'df': np.float32(2.0), 'mel': np.float32(2.0), 'nv': np.float32(1.0), 'vasc': np.float32(2.0)}


## CELL 6: two-stage fine-tune
Same structure as MobileNetV3: freeze everything but the head first (lets
the new classifier settle before backprop disturbs pretrained ViT weights),
then unfreeze the whole backbone at a low LR. ViTs are generally more
sensitive to LR than CNNs, so stage-2 LR here is lower than MobileNetV3's.

In [8]:
# ============================================== CELL 6: two-stage fine-tune
best_f1, best_state = -1, None

# Stage 1 — frozen backbone, head only
head_params = list(model.get_classifier().parameters())
head_param_ids = {id(p) for p in head_params}
for p in model.parameters():
    p.requires_grad_(id(p) in head_param_ids)

opt = torch.optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
run_epochs(5, opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, 5), "S1")

# Stage 2 — unfreeze EVERYTHING, low LR on the backbone (ViTs are touchier
# than CNNs here -- 1e-5 vs MobileNetV3's 1e-4)
for p in model.parameters():
    p.requires_grad_(True)
EPOCHS2 = 25
backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]
opt = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-5},
    {"params": head_params,     "lr": 5e-4},
], weight_decay=1e-4)
run_epochs(EPOCHS2, opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS2), "S2")

model.load_state_dict(best_state)
print(f"\nbest val macro-F1 {best_f1:.4f}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


S1 ep  0  val macro-F1 0.4646  (76s)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


S1 ep  1  val macro-F1 0.5876  (74s)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


...

S2 ep 22  val macro-F1 0.7384  (77s)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


S2 ep 23  val macro-F1 0.7392  (77s)


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


S2 ep 24  val macro-F1 0.7413  (79s)

best val macro-F1 0.7523


## CELL 7: temperature scaling + test evaluation

In [9]:
# ============================== CELL 7: temperature scaling + test evaluation
_, lv, yv = evaluate(dl_va, return_logits=True)
lv, yv = lv.to(DEVICE), yv.to(DEVICE)
logT = torch.zeros(1, device=DEVICE, requires_grad=True)
topt = torch.optim.LBFGS([logT], lr=0.1, max_iter=60)


def _closure():
    topt.zero_grad()
    l = F.cross_entropy(lv / logT.exp(), yv); l.backward(); return l


topt.step(_closure)
TEMP = float(logT.exp())
print(f"fitted temperature T = {TEMP:.3f}")


def ece(probs, labels, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    acc = (pred == labels).astype(np.float32)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

_, lt, yt = evaluate(dl_te, return_logits=True)
yt = yt.numpy()
p_raw = F.softmax(lt, 1).numpy()
p_cal = F.softmax(lt / TEMP, 1).numpy()
pred = p_cal.argmax(1)

MB2 = {"test_acc": accuracy_score(yt, pred),
       "test_macro_f1": f1_score(yt, pred, average="macro"),
       "ece_raw": ece(p_raw, yt), "ece_cal": ece(p_cal, yt),
       "val_macro_f1": best_f1, "temperature": TEMP}

print(f"\ntest accuracy   {MB2['test_acc']:.4f}")
print(f"test macro-F1   {MB2['test_macro_f1']:.4f}")
print(f"ECE  raw {MB2['ece_raw']:.4f}  ->  calibrated {MB2['ece_cal']:.4f}")
print("\n", classification_report(yt, pred, target_names=CLASSES, digits=3))
print(pd.DataFrame(confusion_matrix(yt, pred), index=CLASSES, columns=CLASSES))

torch.save({"state": best_state, "classes": CLASSES, "temperature": TEMP,
            "img_size": IMG_SIZE, "color_constancy": USE_COLOR_CONSTANCY,
            "weight_cap": WEIGHT_CAP, "model_name": MODEL_NAME, "metrics": MB2},
           f"{OUT}/layer2_vit_small.pt")
json.dump({k: float(v) for k, v in MB2.items()},
          open(f"{OUT}/metrics_vit_small.json", "w"), indent=2)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


fitted temperature T = 0.905


/tmp/ipykernel_512/2666535091.py:14: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  TEMP = float(logT.exp())
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



test accuracy   0.8203
test macro-F1   0.6917
ECE  raw 0.0325  ->  calibrated 0.0302

               precision    recall  f1-score   support

       akiec      0.500     0.618     0.553        34
         bcc      0.803     0.685     0.739        89
         bkl      0.679     0.643     0.661       168
          df      0.625     0.556     0.588         9
         mel      0.467     0.623     0.534       146
          nv      0.933     0.902     0.918       974
        vasc      0.895     0.810     0.850        21

    accuracy                          0.820      1441
   macro avg      0.700     0.691     0.692      1441
weighted avg      0.835     0.820     0.826      1441

       akiec  bcc  bkl  df  mel   nv  vasc
akiec     21    2    4   0    6    1     0
bcc        7   61   10   1    6    2     2
bkl       10    3  108   0   17   30     0
df         1    0    2   5    0    1     0
mel        2    6   19   0   91   28     0
nv         1    4   16   2   72  879     0
vasc       0  

## CELL 8: comparison table (loads Layer 1's real saved metrics, not a
hardcoded placeholder)

In [10]:
# ================================================ CELL 8: side-by-side table
mb1_path = f"{OUT}/metrics_mobilenetv3.json"
assert os.path.exists(mb1_path), "run the MobileNetV3 script first (Cell 7 writes this file)"
MB1 = json.load(open(mb1_path))

cmp = pd.DataFrame({
    "MobileNetV3 (fine-tuned)": [MB1["test_acc"], MB1["test_macro_f1"],
                                 MB1["ece_raw"], MB1["ece_cal"], MB1["val_macro_f1"]],
    "ViT-S/16 (fine-tuned)":    [MB2["test_acc"], MB2["test_macro_f1"],
                                 MB2["ece_raw"], MB2["ece_cal"], MB2["val_macro_f1"]],
}, index=["test accuracy", "test macro-F1", "ECE raw", "ECE calibrated", "val macro-F1"])
print(cmp.round(4))
cmp.to_csv(f"{OUT}/layer1_vs_layer2_comparison.csv")

mel_i = CLASSES.index("mel")
cm = confusion_matrix(yt, pred)
print(f"\nmelanoma recall (ViT): {cm[mel_i, mel_i] / cm[mel_i].sum():.3f}")
print(f"melanoma -> nv misses (ViT): {cm[mel_i, CLASSES.index('nv')]}")

                MobileNetV3 (fine-tuned)  ViT-S/16 (fine-tuned)
test accuracy                     0.8446                 0.8203
test macro-F1                     0.7059                 0.6917
ECE raw                           0.0289                 0.0325
ECE calibrated                    0.0249                 0.0302
val macro-F1                      0.7152                 0.7523

melanoma recall (ViT): 0.623
melanoma -> nv misses (ViT): 28


## CELL 9: does the ViT actually help on the cases Layer 1 escalates?
This is the real test for the cascade. `test_routing_detail.csv` comes from
the MobileNetV3 script's threshold sweep (Cell 15) and records exactly
which test images Layer 1 flags as low-confidence. Overall ViT test F1
being good doesn't matter if it's not better than MobileNetV3 specifically
on the escalated subset -- that's `esc_acc` from the sweep, the bar Layer 2
has to clear.

In [11]:
# ==================== CELL 9: score on Layer 1's escalated subset only
routing_path = f"{OUT}/test_routing_detail.csv"
assert os.path.exists(routing_path), (
    "run the MobileNetV3 script's threshold-sweep cells (9-15) first -- "
    "this file records which test images Layer 1 escalates."
)
routing = pd.read_csv(routing_path)

# align routing rows to this script's test dataframe order
te_ids = te.image_id.values
routing = routing.set_index("image_id").loc[te_ids].reset_index()
esc_mask = routing["escalated"].values.astype(bool)

vit_acc_on_escalated = (pred[esc_mask] == yt[esc_mask]).mean()
vit_f1_on_escalated = f1_score(yt[esc_mask], pred[esc_mask], average="macro")
mobilenet_esc_acc = (routing.loc[esc_mask, "pred"] == routing.loc[esc_mask, "true"]).mean()

print(f"escalated subset size: {esc_mask.sum()} / {len(esc_mask)}")
print(f"MobileNetV3 accuracy on escalated subset : {mobilenet_esc_acc:.4f}")
print(f"ViT-S accuracy on escalated subset        : {vit_acc_on_escalated:.4f}")
print(f"ViT-S macro-F1 on escalated subset        : {vit_f1_on_escalated:.4f}")
print()
if vit_acc_on_escalated > mobilenet_esc_acc:
    print("ViT beats MobileNetV3 on the hard cases -- escalation is earning its keep.")
else:
    print("ViT does NOT beat MobileNetV3 on the hard cases -- the cascade "
          "isn't adding value yet. Consider: more epochs, different LR, "
          "or training this model on layer2_train_subset.csv specifically.")

escalated subset size: 426 / 1441
MobileNetV3 accuracy on escalated subset : 0.6291
ViT-S accuracy on escalated subset        : 0.6103
ViT-S macro-F1 on escalated subset        : 0.5580

ViT does NOT beat MobileNetV3 on the hard cases -- the cascade isn't adding value yet. Consider: more epochs, different LR, or training this model on layer2_train_subset.csv specifically.


In [12]:
from scipy.stats import binomtest
# how many of the 426 does each model get right, on the same cases?
mb_correct = (routing.loc[esc_mask, "pred"] == routing.loc[esc_mask, "true"]).values
vit_correct = (pred[esc_mask] == yt[esc_mask])
# paired disagreement: cases where they differ
only_mb_right = (mb_correct & ~vit_correct).sum()
only_vit_right = (~mb_correct & vit_correct).sum()
print(f"MobileNet-only-right: {only_mb_right}  ViT-only-right: {only_vit_right}")
print(binomtest(only_vit_right, only_vit_right + only_mb_right, 0.5))

MobileNet-only-right: 70  ViT-only-right: 62
BinomTestResult(k=62, n=132, alternative='two-sided', statistic=0.4696969696969697, pvalue=0.5425074179193998)
